# nb04 — Single-donor RNA program characterization

## Scientific question
Which donor and biologically comparable cell populations provide a useful test case,
and which transcriptional programs characterize those populations in each capture?

## Analysis strategy
Inspect the benchmark metadata and count layer without loading the full ATAC matrix.
Select a common donor by an explicit coverage/diversity rule; retain original labels
and audit every broad lineage mapping. Normalize RNA counts independently with the
same log1p(CP10k) convention. Characterize mean expression, detection and specificity
against equally weighted other shared cell types. Rank-enrich one cached human-symbol
Hallmark collection against the same shared gene universe.

This is marker/program characterization in healthy cells, **not condition DE**.
CITE and Multiome are separate captures. Cells must never be cross-matched.

**Run location:** your existing Colab/Drive project, as in nb01. Copy the updated
`src/` directory and these notebooks to Drive before running. Local synthetic tests
do not constitute biological validation. nb01–nb03 are preserved.

In [ ]:
from pathlib import Path
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Same Drive root as nb01. No changes to the original benchmark files.
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'anndata>=0.11,<0.13'])
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/multiomics-relationship-modeling')
else:
    BASE_PATH = Path.cwd().resolve()
    if not (BASE_PATH / 'src').is_dir():
        BASE_PATH = BASE_PATH.parent

if not (BASE_PATH / 'src/single_donor_workflow.py').is_file():
    raise FileNotFoundError('Copy the updated src/, configs/ and new notebooks to the project on Drive first.')
sys.path.insert(0, str(BASE_PATH))
import pandas as pd
from IPython.display import display, Image
from configs.paths import CITE_H5AD, MULTIOME_H5AD
from src.single_donor_io import output_dirs

OUTPUT_ROOT = BASE_PATH / 'results/single_donor'
DIRS = output_dirs(OUTPUT_ROOT)
print('Colab:', IN_COLAB, '| Project:', BASE_PATH)

## Parameters and input inspection
Defaults require at least 50 cells per type in each capture, at least three supported
shared types, and no population above 70% of either donor capture. These development
thresholds are explicit, configurable and saved. No donor is hard-coded.

The count layer is used because `.X` has uncertain/incompatible processed scales in
the repository's DATA.md. RNA library size is calculated on **GEX only**, never ADT or
ATAC. Each capture retains its own GEX panel during normalization; panel differences
are a limitation. Existing preprocessing and input files are not overwritten.

In [ ]:
from src.single_donor_io import inspect_h5ad

PATHS = {'cite': CITE_H5AD, 'multiome': MULTIOME_H5AD}
MIN_CELLS = 50
MIN_SHARED_TYPES = 3
MAX_DOMINANCE = 0.70
RANDOM_SEED = 42
PERMUTATIONS = 1000
MIN_DETECTION = 0.01
COMPUTE_MEDIANS = False  # optional blockwise sparse->dense calculation
MAPPING_OVERRIDE = None  # path to a reviewed copy of celltype_mapping.csv

for dataset, path in PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(f'{dataset}: missing {path}; use the same data/benchmark location as nb01.')
    obs, var, audit = inspect_h5ad(path)
    print(dataset, audit)
    display(obs.groupby(['DonorID', 'Site'], observed=True).size().rename('n_cells').reset_index())
    print('Original labels:', sorted(obs.cell_type.unique()))
del obs, var

## Donor selection and explicit cell-type harmonization
Candidates are ranked by eligible shared-type coverage, sum of minimum per-type cell
counts across captures, lower maximum dominance, then donor ID. Dominance includes
unresolved cells. The full donor ranking records the criteria and selection.

Mappings are explicit code rules, not fuzzy label matching. CD4/CD8 subtypes, B-cell
subtypes and erythroid stages are pooled as documented. HSC, lymphoid, G/M and MK/E
progenitors remain separate; pDC and cDC populations remain separate. Labels such as
T reg, MAIT, gamma-delta T, ILC1 and plasmablasts are not silently assigned to broader
comparators. Exact otherwise-unmapped labels can be retained if present in both.

**Review the mapping and subtype composition before interpreting results.** To revise
it, copy `celltype_mapping.csv`, edit the harmonized labels/confidence/reasons, set
`MAPPING_OVERRIDE` above, and rerun. Do not edit a saved result in place and continue
with stale downstream outputs. Unresolved or undersized populations remain in audits
but are excluded from the shared RNA reference.

In [ ]:
from src.single_donor_workflow import prepare_donor

donor, donor_ranking, cell_counts, rna_data = prepare_donor(
    PATHS, OUTPUT_ROOT, min_cells=MIN_CELLS, min_shared=MIN_SHARED_TYPES,
    max_dominance=MAX_DOMINANCE, mapping_override=MAPPING_OVERRIDE)
print('Automatically selected donor:', donor)
display(donor_ranking)
display(pd.read_csv(DIRS['donor_selection'] / 'celltype_mapping_paired.csv'))
display(pd.read_csv(DIRS['donor_selection'] / 'selected_donor_mapping.csv'))
display(cell_counts)
display(Image(filename=str(DIRS['donor_selection'] / 'figures/cell_counts.png')))

## Independent RNA profiles and ranked enrichment
Mean expression is the mean of log1p(CP10k), not log of an aggregated count total.
Specificity is target-type mean minus the equally weighted mean of other retained
types. All types use the same reference populations in both datasets.

Mitochondrial, ribosomal and selected housekeeping genes are **flagged**, retained in
complete rankings, and excluded only from the compact marker figure where indicated.
Technical spike-ins are excluded from enrichment. Shared genes must be detected in
at least 1% of cells in at least one retained type in **each** capture; this creates
one fixed, symmetric universe, not separate cell-type-specific gene filters.

The default collection is Enrichr's `MSigDB_Hallmark_2020` human-symbol library (50
sets). The first run downloads it explicitly; subsequent runs reuse the exact GMT.
Its SHA256 and bytes are saved. A locally supplied human-symbol Hallmark, Reactome or
GO-BP GMT may be substituted, but both datasets must use the same file. No fabricated
gene sets or automatic database substitution are used on network failure.

The dependency-light GSEA-style implementation uses a weighted running-sum statistic
(p=1), same-sign gene-set permutation nulls, NES, and plus-one nominal p-values.
`q_value` is **Benjamini–Hochberg within dataset × cell type**, not Broad's pooled GSEA
FDR. An additional pooled BH column is saved in `all_enrichment.csv`. Null gene sets
ignore gene correlations, so significance is exploratory. No individual-level
replication is implied. Stable gene-symbol ordering breaks rank ties; tie fractions
and gene-set coverage are reported. Sets need 15–500 overlapping genes and fewer than
the full ranking. Negative enrichment means lower **relative specificity**, not
absence/inactivity of a biological pathway.

Method reference: [GSEA User Guide](https://docs.gsea-msigdb.org/GSEA/GSEA_User_Guide/).

In [ ]:
from src.pathway_enrichment import cache_hallmark
from src.single_donor_workflow import characterize_donor

GMT_PATH = DIRS['gsea'] / 'MSigDB_Hallmark_2020.gmt'
# For a custom GMT, replace GMT_PATH and omit cache_hallmark below.
print('Cached collection SHA256:', cache_hallmark(GMT_PATH))
programs, enrichment, universe = characterize_donor(
    rna_data, OUTPUT_ROOT, GMT_PATH, permutations=PERMUTATIONS,
    seed=RANDOM_SEED, detection=MIN_DETECTION, median=COMPUTE_MEDIANS)
print('Shared eligible genes:', len(universe))
display(pd.read_csv(DIRS['gsea'] / 'gene_set_coverage.csv'))
for dataset in ['cite', 'multiome']:
    print(dataset)
    display(programs[dataset].sort_values('specificity', ascending=False).groupby('cell_type').head(5))
    display(enrichment[dataset].sort_values(['cell_type', 'q_value']).groupby('cell_type').head(5))
    display(Image(filename=str(DIRS[f'rna_{dataset}'] / 'figures/characteristic_genes.png')))
    figure = DIRS[f'rna_{dataset}'] / 'figures/pathway_enrichment.png'
    if not enrichment[dataset].empty and figure.exists():
        display(Image(filename=str(figure)))

## Main findings
The next cell reports observed counts and test coverage only. Inspect the saved
rankings and leading-edge genes before writing a biological interpretation. No
biological finding is pre-filled in this unexecuted notebook.

In [ ]:
print(f'Donor {donor}: {int(cell_counts.included.sum())} retained shared populations; {len(universe)} comparison genes.')
for dataset, result in enrichment.items():
    print(dataset, ':', len(result), 'pathway/type tests;', int((result.q_value <= 0.05).sum()),
          'with within-ranking BH q <= 0.05 (exploratory).')
print('Next: run nb05 to test concordance and sampling/site sensitivity before any multimodal follow-up.')

## Limitations
- One healthy donor: no disease/control contrast and no inference across individuals.
- Separate captures: broad cell types may contain different state/subtype mixtures.
- Unequal capture sizes are reported; nb05 performs independent balanced subsampling.
- Site composition may confound pooled donor summaries; nb05 checks shared sites.
- Library normalization uses slightly different measured GEX panels; equal processing
  cannot remove capture, depth, ambient RNA or annotation differences.
- This workflow uses organizer-processed cells with no new QC threshold imposed;
  examine nb01's QC findings and depth summaries before interpretation.
- Gene-set p-values are competitive, exploratory statistics, not donor-level tests.
- Later ADT evidence will cover a limited panel; unmeasured proteins are not negatives.
- Later ATAC gene activity is an approximation, and cross-sectional data cannot prove
  ATAC → RNA → protein causality or temporal priming.